# VisDrone — 2/4: **Ours** (prune 50% + CWD)

L1-norm uniform 50% (divisor 8) tu baseline cua notebook 1, roi finetune 100
epoch voi CWD distillation (tau=9, kd_layers=neck, kd_warmup=5).

| | |
|---|---|
| Nguon prune + teacher | `best.pt` cua **nb1_yolo26m** |
| Epoch / batch / imgsz | 100 / 16 / 640 |
| Uoc tinh | ~8-9h, **2 phien** |

> **Phai doi notebook 1 chay xong truoc.** Add Data output cua nb1 vao day.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
2. **Add Data** -> output cua nb1 (chua `runs/vd_yolo26m/weights/best.pt`).
   Bam **Save & Run All**.
3. Phien tu dung o 10h. Cell cuoi bao `CHUA XONG` kem so epoch -> Add Data
   output cua chinh lan chay nay roi Save & Run All lai. Lap den khi bao `XONG`.
4. Xong thi gui lai 3 so o cell cuoi (Params / AP50 / AP50-95).

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup

In [ ]:
import os, sys, glob, shutil, pathlib, subprocess

REPO_DIR = pathlib.Path("/kaggle/working/yolo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Phai dung fork nay, KHONG "pip install ultralytics": checkpoint sau khi prune
# duoc pickle voi ultralytics.nn.tasks_pruned nen ban chinh thuc khong load duoc.
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "pruning"))
# DDP sinh tien trinh con chay file tam ngoai repo -> phai truyen qua PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "--no-deps"], check=False)

import torch
from ultralytics import YOLO
print("GPU:", torch.cuda.device_count())

## 2. Cau hinh

In [ ]:
DATA   = "VisDrone.yaml"   # Ultralytics tu tai 2.3 GB, can Internet: On
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640
DEVICE = "0,1" if torch.cuda.device_count() > 1 else "0"
STOP_AFTER_H = 10.0        # tu dung truoc moc 12h de output kip luu

NAME  = "vd_ours50"
RATIO = 0.5

# Baseline VisDrone tu notebook 1. Tim trong /kaggle/input.
TEACHER = None
for c in glob.glob("/kaggle/input/**/runs/vd_yolo26m/weights/best.pt", recursive=True):
    TEACHER = c
    break
print("teacher:", TEACHER or "!! CHUA CO - Add Data output cua nb1 vao")

## 3. Prune 50%

In [ ]:
# Goi thang cac ham trong pruning/, khong qua script.
from prune_common import load_and_prepare, create_masks, finalize_pruning
from prune_l1norm import compute_l1norm_importance

assert TEACHER, "Chua co baseline cua nb1"
PRUNED = REPO_DIR / "weights" / "yolo26m_vd_pruned50.pt"

if PRUNED.exists():
    print("da co", PRUNED)
else:
    PRUNED.parent.mkdir(parents=True, exist_ok=True)
    m0, bn_dict, ignore_bn, _chunk, layer_cfg, pruned_yaml = load_and_prepare(
        TEACHER, str(REPO_DIR / "cfg" / "yolo26m.yaml"), "m", None)
    imp = compute_l1norm_importance(m0, bn_dict, ignore_bn)
    masks = create_masks(imp, m0, ignore_bn, layer_cfg, RATIO, 8)
    out = finalize_pruning(m0, masks, pruned_yaml, ignore_bn, TEACHER,
                           str(REPO_DIR / "weights"), 8, RATIO,
                           method_name="l1norm")
    pathlib.Path(out).replace(PRUNED)
    print("->", PRUNED)

## 4. Resume

In [ ]:
# Kaggle giet phien o 12h. Moi phien train toi da STOP_AFTER_H roi tu dung va
# luu last.pt. Lan sau: Add Data -> Your Work -> output lan truoc, cell nay chep
# runs/ ve roi train tiep.
run_dir = REPO_DIR / "runs" / NAME
if not run_dir.exists():
    for src in glob.glob("/kaggle/input/**/runs/" + NAME, recursive=True):
        if pathlib.Path(src, "weights", "last.pt").exists():
            shutil.copytree(src, run_dir)
            print("chep ve tu", src)
            break

last = run_dir / "weights" / "last.pt"
print("co last.pt -> train tiep" if last.exists() else "chua co -> train tu dau")

## 5. Finetune + CWD

In [ ]:
if last.exists():
    model = YOLO(str(last))
    model.train(resume=True, stop_after_h=STOP_AFTER_H)
else:
    model = YOLO(str(PRUNED))
    model.train(data=DATA, epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
                device=DEVICE, seed=0,
                project=str(REPO_DIR / "runs"), name=NAME, exist_ok=True,
                stop_after_h=STOP_AFTER_H,
                finetune=True,                 # build DetectionModelPruned tu maskbndict
                kd=True, kd_teacher=TEACHER, kd_method="cwd",
                kd_lambda=0.5, kd_layers="neck", kd_warmup=5,
                cwd_temperature=9.0)

## 6. Ket qua

In [ ]:
import pandas as pd

csv = run_dir / "results.csv"
df = pd.read_csv(csv)
df.columns = df.columns.str.strip()
ep = int(df["epoch"].max())
print("epoch {}/{}".format(ep, EPOCHS))

if ep >= EPOCHS:
    m = YOLO(str(run_dir / "weights" / "best.pt"))
    r = m.val(data=DATA, imgsz=IMGSZ, batch=BATCH, device=DEVICE.split(",")[0])
    n_par = sum(p.numel() for p in m.model.parameters()) / 1e6
    print()
    print("  Params   {:.2f} M".format(n_par))
    print("  AP50     {:.2f}".format(r.box.map50 * 100))
    print("  AP50-95  {:.2f}".format(r.box.map * 100))
    print()
    print("XONG. Gui lai 3 so tren + duong dan:")
    print("  ", run_dir / "weights" / "best.pt")
else:
    print()
    print("CHUA XONG - Add Data output lan nay roi Save & Run All lai.")